<a href="https://colab.research.google.com/github/maliaheupel/ds2002-fa2/blob/main/05-etl-walmart/2026_10_16_%E2%80%94_Walmart_Warm_Up_SKU_Mess_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [5]:
# TODO (hint: .str.lower().str.contains('pop'))
poptarts = cat[cat['product_name'].str.lower().str.contains('pop')]

print(f'Rows: {len(poptarts)} ')
poptarts

Rows: 3 


,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [6]:
# TODO
strawberries = poptarts[poptarts['product_name'].str.lower().str.contains('strawberry')]
print(f'Strawberry rows: {len(strawberries)} ')
print(f'Non-strawberry rows: {len(poptarts) - len(strawberries)} ')
strawberries

Strawberry rows: 3 
Non-strawberry rows: 0 


,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


**Pop-Tart rows that are a different product:** need to stay differentiated based on flavors. If not, it will distract from any flavor-specific averages or sales indicators.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [8]:
# TODO
cannon = {
    'PT-12': 'SKU#459812',
    'SKU#459812': 'SKU#459812',
    'POPTART-STRAW': 'SKU#459812'
}
cat['canonical'] = cat['sku'].map(cannon).fillna(cat['sku'])
cat


,sku,product_name,category,canonical
0,PT-12,pop-tarts strawberry,Breakfast Foods,SKU#459812
1,SKU#459812,Strawberry Pop-Tart,Breakfast,SKU#459812
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,SKU#459812


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [9]:
# TODO
print('Before:', cat['category'].value_counts())

cat['category'] = cat['category'].str.strip().str.title()

print('After:', cat['category'].value_counts())

Before: category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64
After: category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [11]:
# TODO
cat['normalized'] = (cat['product_name'].str.lower()
                     .str.replace(r'[^a-z0-9 ]', ' ', regex=True)
                     .str.split().apply(lambda w: ' '.join(sorted(w))))

candidates = (cat.groupby('normalized')['sku']
              .agg(list).loc[lambda s: s.str.len() > 1])

candidates

,sku
normalized,
pop strawberry tarts,"[PT-12, POPTART-STRAW]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [13]:
# TODO
assert cat['canonical'].notna().all()
assert len(cat) == 3
print("all checks passed")

all checks passed


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) Two more fragmented product families were water bottles and flashlights. I would group them together similarly to how I did so with poptarts but within their own categories. This way their quantities would be reflective of an accurate, true count.

b) Fixng things like merging duplicates are a data-cleaning task. However when deciding to merge products like a 12-count of poptarts with single poptarts would be more of a business decision. In the real world, if this kind of issue arose, I would ask someone in charge of product strategy who knows how sales and the supply are actually evaluated.